# NB09: LeakGuard - suspicious column and correlation filter

Verifies that no embedding dimension carries direct label information.
Drops any column whose name suggests a label leak, then drops any column
whose absolute Pearson correlation with HRD_top20 on the training split
exceeds 0.98. Refits the pipeline if any columns are dropped and compares
the resulting val/test AUROC to NB07.

**Manuscript reference** (Supplementary Table S8):
- Columns dropped (missing): 0
- Columns dropped (constant): 0
- Imputed cells: 0
- Final dimensions: 512

For OpenCLIP ViT-B/16 the feature columns are anonymous (`f000`..`f511`)
so the name-based filter has nothing to remove. The correlation filter
acts as a structural sanity check; the manuscript reports 0 drops here,
so the refit results should match NB07 exactly. Any divergence is
flagged so it can be investigated before publication.

**Required env**: `WORKSPACE`. **Inputs**: clean embeddings + labels.
**Outputs**: `results/leakguard/{report.json, dropped_columns.csv}`.

In [ ]:
import os
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

# env
WORKSPACE = Path(os.environ.get("WORKSPACE", "./workspace")).resolve()
EMB_DIR = WORKSPACE / "embeddings"
LABELS_DIR = WORKSPACE / "labels"
RESULTS_DIR = WORKSPACE / "results" / "leakguard"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# manuscript-locked params
SEED = 42
PCA_N = 384
RIDGE_ALPHA = 30.0
HRD_THR = 33
CORR_BAN = 0.98
SUSPICIOUS_NAME_PATTERNS = [
    r"\bhrd\b", r"label", r"target", r"^y_", r"_y$",
    r"scarhrd", r"loh", r"\btai\b", r"\blst\b", r"split",
]

print(f"corr_ban={CORR_BAN}, name patterns={SUSPICIOUS_NAME_PATTERNS}")


In [ ]:
# load aligned embeddings + labels with non-null HRD
X = pd.read_parquet(EMB_DIR / "patient_means_clean.parquet").set_index("patient")
labels = pd.read_parquet(LABELS_DIR / "labels.parquet")
labels["patient"] = labels["patient"].astype(str).str.upper().str.slice(0, 12)
labels = labels.drop_duplicates("patient").set_index("patient")

common = sorted(set(X.index) & set(labels.index))
X = X.loc[common]
labels = labels.loc[common].copy()
labels = labels.loc[labels["HRD"].notna()].copy()
X = X.loc[labels.index]
labels["HRD_top20"] = (labels["HRD"] >= HRD_THR).astype(int)

idx_tr = labels.index[labels["split"] == "train"]
idx_va = labels.index[labels["split"] == "val"]
idx_te = labels.index[labels["split"] == "test"]
print(f"patients with HRD: {len(labels):,}  (train={len(idx_tr)}, val={len(idx_va)}, test={len(idx_te)})")
print(f"feature dim      : {X.shape[1]}")


In [ ]:
# 1) name-based filter
def name_is_suspicious(name, patterns):
    s = str(name).lower()
    return any(re.search(p, s) for p in patterns)

drop_by_name = [c for c in X.columns if name_is_suspicious(c, SUSPICIOUS_NAME_PATTERNS)]
print(f"name-flagged columns: {len(drop_by_name)} (expect 0 for OpenCLIP anonymous f000..f511)")
if drop_by_name:
    print(f"  examples: {drop_by_name[:10]}")


In [ ]:
# 2) correlation filter on TRAIN only (avoid using val/test in any selection step)
X_tr_full = X.loc[idx_tr].drop(columns=drop_by_name)
y_tr_bin = labels.loc[idx_tr, "HRD_top20"].astype(int).values
y_tr_cont = labels.loc[idx_tr, "HRD"].astype(float).values

# Pearson correlation of each column with binary target
def col_corr(arr, y):
    arr_z = (arr - arr.mean(axis=0)) / (arr.std(axis=0, ddof=0) + 1e-12)
    y_z = (y - y.mean()) / (y.std(ddof=0) + 1e-12)
    return arr_z.T @ y_z / len(y)

r_bin = col_corr(X_tr_full.values.astype(np.float64), y_tr_bin.astype(np.float64))
r_cont = col_corr(X_tr_full.values.astype(np.float64), y_tr_cont.astype(np.float64))

drop_by_corr_bin = [c for c, r in zip(X_tr_full.columns, r_bin) if abs(r) >= CORR_BAN]
drop_by_corr_cont = [c for c, r in zip(X_tr_full.columns, r_cont) if abs(r) >= CORR_BAN]
drop_by_corr = sorted(set(drop_by_corr_bin) | set(drop_by_corr_cont))

print(f"max |r| with HRD_top20 (TRAIN): {np.nanmax(np.abs(r_bin)):.3f}")
print(f"max |r| with HRD continuous (TRAIN): {np.nanmax(np.abs(r_cont)):.3f}")
print(f"corr-flagged columns: {len(drop_by_corr)} (expect 0 per Supp S8)")
if drop_by_corr:
    print(f"  examples: {drop_by_corr[:10]}")


In [ ]:
# combined drop set; refit if any drops occurred
all_drops = sorted(set(drop_by_name) | set(drop_by_corr))
keep_cols = [c for c in X.columns if c not in all_drops]
print(f"total dropped: {len(all_drops)}  | kept: {len(keep_cols)}")

X_kept = X[keep_cols]

X_tr = X_kept.loc[idx_tr].values.astype(np.float32)
X_va = X_kept.loc[idx_va].values.astype(np.float32)
X_te = X_kept.loc[idx_te].values.astype(np.float32)
y_va_bin = labels.loc[idx_va, "HRD_top20"].astype(int).values
y_te_bin = labels.loc[idx_te, "HRD_top20"].astype(int).values

# refit identical pipeline to NB07 on the kept columns
pca_n = min(PCA_N, X_tr.shape[1] - 1, X_tr.shape[0] - 1)
pipe = Pipeline([
    ("scaler", StandardScaler(with_mean=True, with_std=True)),
    ("pca", PCA(n_components=pca_n, random_state=SEED)),
    ("ridge", Ridge(alpha=RIDGE_ALPHA, random_state=SEED)),
])
pipe.fit(X_tr, y_tr_cont)
z_tr = pipe.predict(X_tr).reshape(-1, 1)
platt = LogisticRegression(max_iter=1000, solver="lbfgs", random_state=SEED).fit(z_tr, y_tr_bin)

def predict(Xq):
    return platt.predict_proba(pipe.predict(Xq).reshape(-1, 1))[:, 1]

p_va = predict(X_va)
p_te = predict(X_te)

leakguard_metrics = {
    "val_AUROC": float(roc_auc_score(y_va_bin, p_va)),
    "val_AP": float(average_precision_score(y_va_bin, p_va)),
    "val_Brier": float(brier_score_loss(y_va_bin, p_va)),
    "test_AUROC": float(roc_auc_score(y_te_bin, p_te)),
    "test_AP": float(average_precision_score(y_te_bin, p_te)),
    "test_Brier": float(brier_score_loss(y_te_bin, p_te)),
}
print()
print("LEAKGUARD-REFIT METRICS:")
for k, v in leakguard_metrics.items():
    print(f"  {k:<14}= {v:.4f}")


In [ ]:
# compare to NB07 metrics if available
nb07_path = WORKSPACE / "results" / "tcga_internal" / "report.json"
nb07 = json.loads(nb07_path.read_text()) if nb07_path.exists() else None

if nb07 is None:
    print("(NB07 report not found; comparison skipped)")
    diffs = None
else:
    ref_va = nb07["metrics"]["val"]
    ref_te = nb07["metrics"]["test"]
    diffs = {
        "delta_val_AUROC":  leakguard_metrics["val_AUROC"]  - ref_va["AUROC"],
        "delta_val_AP":     leakguard_metrics["val_AP"]     - ref_va["AP"],
        "delta_val_Brier":  leakguard_metrics["val_Brier"]  - ref_va["Brier"],
        "delta_test_AUROC": leakguard_metrics["test_AUROC"] - ref_te["AUROC"],
        "delta_test_AP":    leakguard_metrics["test_AP"]    - ref_te["AP"],
        "delta_test_Brier": leakguard_metrics["test_Brier"] - ref_te["Brier"],
    }
    print()
    print("DELTAS vs NB07 (expected ~0 since manuscript Supp S8 says 0 cols dropped):")
    for k, v in diffs.items():
        print(f"  {k:<20}= {v:+.5f}")


In [ ]:
# write report and dropped-columns csv
pd.DataFrame({
    "column": all_drops,
    "reason": (["name"] * len(drop_by_name)
               + ["corr_bin"] * len([c for c in drop_by_corr if c in drop_by_corr_bin])
               + ["corr_cont"] * len([c for c in drop_by_corr if c in drop_by_corr_cont]))[:len(all_drops)],
}).to_csv(RESULTS_DIR / "dropped_columns.csv", index=False)

report = {
    "seed": SEED,
    "params": {
        "PCA_N": PCA_N, "RIDGE_ALPHA": RIDGE_ALPHA,
        "HRD_THR": HRD_THR, "CORR_BAN": CORR_BAN,
    },
    "input_dim": int(X.shape[1]),
    "kept_dim": int(len(keep_cols)),
    "n_dropped_total": int(len(all_drops)),
    "n_dropped_by_name": int(len(drop_by_name)),
    "n_dropped_by_corr_bin": int(len(drop_by_corr_bin)),
    "n_dropped_by_corr_cont": int(len(drop_by_corr_cont)),
    "max_abs_corr_bin_train": float(np.nanmax(np.abs(r_bin))),
    "max_abs_corr_cont_train": float(np.nanmax(np.abs(r_cont))),
    "leakguard_refit_metrics": leakguard_metrics,
    "deltas_vs_nb07": diffs,
    "manuscript_refs": {
        "cols_dropped_missing": 0,
        "cols_dropped_constant": 0,
        "input_dim": 512,
        "final_dim": 512,
    },
}
(RESULTS_DIR / "report.json").write_text(json.dumps(report, indent=2, default=str))
print(json.dumps(report, indent=2, default=str))
